# Milestone 13 — Prepare the failure-review packet

Run this only after the frozen final comparison. It reuses the selected checkpoints and cached Qwen test scores, verifies the cached rankings with a strict recorded CPU/CUDA drift bound, and samples 80 genuine CompleteRecall@5 failures for human classification. It does not train or tune any model.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from io import BytesIO
import hashlib
import json
import subprocess
import sys

drive.mount('/content/drive')
uploaded = files.upload()  # Select finevid_error_analysis_source.zip.
if len(uploaded) != 1:
    raise ValueError('Upload exactly one error-analysis source bundle.')
bundle_name, bundle_bytes = next(iter(uploaded.items()))
bundle_hash = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT_DIR = Path('/content') / ('finevid-error-analysis-' + bundle_hash[:12])
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(bundle_bytes)) as archive:
    manifest = json.loads(archive.read('bundle_manifest.json'))
    if manifest.get('purpose') != 'FinEvid-Distill Milestone 13 failure-analysis bundle':
        raise ValueError('This is not the Milestone 13 bundle.')
    expected = set(manifest['files']) | {'bundle_manifest.json'}
    if len(archive.namelist()) != len(expected) or set(archive.namelist()) != expected:
        raise ValueError('Bundle file list does not match its manifest.')
    for name, expected_hash in manifest['files'].items():
        destination = (PROJECT_DIR / name).resolve()
        if not destination.is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError('Invalid archive path.')
        if hashlib.sha256(archive.read(name)).hexdigest() != expected_hash:
            raise ValueError('Bundle integrity check failed: ' + name)
    archive.extractall(PROJECT_DIR)
del uploaded, bundle_bytes
ARTIFACT_ROOT = Path('/content/drive/MyDrive/FinEvid-Distill')
TEACHER_CACHE = ARTIFACT_ROOT / 'teacher_scores/teacher_test_scores.jsonl'
HARD_DIR = ARTIFACT_ROOT / 'checkpoints/hard_label_student_tau005'
DISTILLED_DIR = ARTIFACT_ROOT / 'checkpoints/distilled_student'
RESULT_DIR = ARTIFACT_ROOT / 'experiment_results'
SCORE_CACHE_DIR = RESULT_DIR / 'error_analysis_score_cache'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
for required in (TEACHER_CACHE, HARD_DIR, DISTILLED_DIR):
    if not required.exists():
        raise FileNotFoundError(required)
print('Verified Milestone 13 bundle:', bundle_hash)

## Install and verify

A GPU is faster, but a CPU runtime is valid because Qwen logits are already cached. Per-model score files are saved to Drive, so a disconnected rerun resumes at the next model.

In [ ]:
%pip install -q -r {PROJECT_DIR / 'requirements-colab.txt'}
%pip install -q -e {PROJECT_DIR}

In [ ]:
def run_project(*arguments):
    return subprocess.run([sys.executable, '-u', *arguments], cwd=PROJECT_DIR, check=True)

run_project('-m', 'pytest', '-q')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Scoring device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

## Reproduce rankings and sample 80 failures

The sampler selects 20 genuine CompleteRecall@5 failures for each of Frozen BGE, Hard-label BGE, Distilled BGE, and Qwen. It prioritizes hard-versus-distilled contrast cases, then uses a seed-42 ordering for the remainder.

In [ ]:
PACKET = RESULT_DIR / 'error_review_packet.json'
run_project(
    'src/evaluation/prepare_error_analysis.py',
    '--selection', str(PROJECT_DIR / 'outputs/final_selection.json'),
    '--final-results', str(PROJECT_DIR / 'outputs/final_test_results.json'),
    '--test-data', str(PROJECT_DIR / 'data/processed/test.jsonl'),
    '--raw-test-data', str(PROJECT_DIR / 'data/raw/test.json'),
    '--teacher-cache', str(TEACHER_CACHE),
    '--hard-dir', str(HARD_DIR),
    '--distilled-dir', str(DISTILLED_DIR),
    '--score-cache-dir', str(SCORE_CACHE_DIR),
    '--output', str(PACKET),
    '--device', DEVICE,
    '--batch-size', '64' if DEVICE == 'cuda' else '16',
    '--failures-per-model', '20',
)
packet = json.loads(PACKET.read_text())
assert packet['review_instance_count'] == 80
print('Locked full-test failure counts:')
print(json.dumps(packet['full_test_failure_counts'], indent=2))
print('Review-device metric deltas from the locked final result:')
print(json.dumps(packet['metric_deltas_from_locked_final'], indent=2))

## Download the compact packet for human classification

In [ ]:
review_zip = Path('/content/milestone13_review_packet.zip')
with ZipFile(review_zip, 'w', compression=ZIP_DEFLATED) as archive:
    archive.write(PACKET, 'error_review_packet.json')
    archive.writestr('source_bundle_sha256.txt', bundle_hash)
    archive.writestr(
        'score_cache_sha256.json',
        json.dumps({path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in sorted(SCORE_CACHE_DIR.glob('*.jsonl'))}, indent=2, sort_keys=True),
    )
files.download(str(review_zip))